In [1]:
import os
import glob
import librosa
import soundfile as sf
import numpy as np
import pandas as pd

In [2]:

# CONFIG
INPUT_FOLDER = 'D:/AIS/Action Learning/repo/birdclef_2026/data/train_soundscapes'
OUTPUT_FOLDER = 'D:/AIS/Action Learning/repo/birdclef_2026/data/train_soundscapes_chunks/'
LABELS_CSV_PATH = 'D:/AIS/Action Learning/repo/birdclef_2026/data/train_soundscapes_labels.csv'
CHUNK_DURATION = 5  # seconds
TARGET_SR = 32000   # standard sampling rate for BirdCLEF models (use None to keep original)

In [3]:
labels_df = pd.read_csv(LABELS_CSV_PATH)
# Get a unique set of labeled filenames
labeled_filenames = set(labels_df['filename'].str.replace('.ogg', '', regex=False).unique())
print(f"Loaded labeled CSV. Found {len(labeled_filenames)} unique labeled soundscape files.")

# Find all audio files (BirdCLEF usually uses .ogg)
audio_files = glob.glob(os.path.join(INPUT_FOLDER, "*.ogg"))

print(f"Found {len(audio_files)} audio files to split.")

audio_files = audio_files

Loaded labeled CSV. Found 66 unique labeled soundscape files.
Found 10658 audio files to split.


In [4]:
unlabeled_records = []

# Ensure the output directory is created before writing files
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

count = 0

for file_path in audio_files:
    count = count + 1
    if count % 1000 == 0: 
        print(f"File number {count}")

    
    # Extract file name without extension
    base_name = os.path.splitext(os.path.basename(file_path))[0]

    # Load audio file 
    y, sr = librosa.load(file_path, sr=TARGET_SR)
    
    # Calculate total duration and total number of samples per 5-second chunk
    total_samples = len(y)
    chunk_samples = CHUNK_DURATION * sr
    
    for start_sample in range(0, total_samples, chunk_samples):
        end_sample = start_sample + chunk_samples
        
        # Slice the audio array
        chunk_y = y[start_sample:end_sample]
        
        # Skip the last snippet if it is completely empty or heavily truncated
        if len(chunk_y) < (sr * 1): # skips chunks shorter than 1 second
            continue
            
        # Calculate timeline bounds for naming conventions
        start_sec = int(start_sample / sr)
        end_sec = int(start_sec + CHUNK_DURATION)
        
        # Construct output file name (e.g., soundscape_file_5_10.ogg)
        output_file_name = f"{base_name}_{start_sec}.ogg"

        if base_name in labeled_filenames:
            print(f"Found {output_file_name}. Skipping it")
            continue
        unlabeled_records.append(output_file_name)
        
        # Save chunk
        sf.write(OUTPUT_FOLDER+output_file_name, chunk_y, sr, format='OGG', subtype='VORBIS')

print(f"All files have been successfully split into 5-second chunks and saved to: {OUTPUT_FOLDER}")

Found BC2026_Train_0001_S08_20250606_030007_0.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_5.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_10.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_15.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_20.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_25.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_30.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_35.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_40.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_45.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_50.ogg. Skipping it
Found BC2026_Train_0001_S08_20250606_030007_55.ogg. Skipping it
Found BC2026_Train_0002_S08_20250607_030007_0.ogg. Skipping it
Found BC2026_Train_0002_S08_20250607_030007_5.ogg. Skipping it
Found BC2026_Train_0002_S08_20250607_030007_10.ogg. Skipping it
Found BC2026_Train_0002_S08_20250607_030007_

In [5]:
df = pd.DataFrame(unlabeled_records, columns=["filename"])

# Save to CSV
df.to_csv("unlabeled_soundscape_chunks.csv", index=False)

print("CSV created using Pandas!")

CSV created using Pandas!
